# Análise Exploratória de Dados (EDA)

## 1. Introdução 

Este projeto realiza uma Análise Exploratória de Dados (EDA) sobre corridas da Uber com o objetivo de compreender padrões de utilização, comportamento temporal das corridas e possíveis aplicações de modelos de Inteligência Artificial para previsão de demanda.

**Objetivo da análise:**

- entender padrões de uso da Uber

- analisar variáveis do dataset

- identificar padrões temporais

- verificação o que leva uma pessoa cancelar uma corrida 

- gerar visualizações

- indicar possíveis aplicações de IA

In [1]:
!pip install missingno

## 2. Importação das bibliotecas

In [2]:
# importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
import os

warnings.filterwarnings("ignore")
plt.style.use("ggplot")

## 3. Carregamento dos dados 

In [3]:
df = pd.read_csv("../data/raw/ncr_ride_bookings.csv")

In [4]:
df.shape

(150000, 21)

In [5]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


## 4. Entendimento inicial dos dados

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

In [7]:
df.describe()

,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Cancelled Rides by Driver,Incomplete Rides,Booking Value,Ride Distance,Driver Ratings,Customer Rating
count,139500.000000,102000.000000,10500.0,27000.0,9000.0,102000.000000,102000.000000,93000.000000,93000.000000
mean,8.456352,29.149636,1.0,1.0,1.0,508.295912,24.637012,4.230992,4.404584
std,3.773564,8.902577,0.0,0.0,0.0,395.805774,14.002138,0.436871,0.437819
min,2.000000,10.000000,1.0,1.0,1.0,50.000000,1.000000,3.000000,3.000000
25%,5.300000,21.600000,1.0,1.0,1.0,234.000000,12.460000,4.100000,4.200000
50%,8.300000,28.800000,1.0,1.0,1.0,414.000000,23.720000,4.300000,4.500000
75%,11.300000,36.800000,1.0,1.0,1.0,689.000000,36.820000,4.600000,4.800000
max,20.000000,45.000000,1.0,1.0,1.0,4277.000000,50.000000,5.000000,5.000000


In [8]:
df.columns

Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT',
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Payment Method'],
      dtype='object')

## 5. Dicionário dos dados

In [9]:
# Criação do dicionário de dados
data_dictionary = pd.DataFrame([
    {
        'variable': 'Date',
        'description': 'Date of the booking',
        'type': 'temporal',
        'subtype': 'date',
    },
    {
        'variable': 'Time',
        'description': 'Time of the booking',
        'type': 'temporal',
        'subtype': 'time',
    },
     {
        'variable': 'Booking ID',
        'description': 'Unique identifier for each ride booking',
        'type': 'qualitative',
        'subtype': 'identifier',
    },
     {
        'variable': 'Booking Status',
        'description': 'Status of booking (Completed, Cancelled by Customer, Cancelled by Driver, etc.)',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
    {
        'variable': 'Customer ID',
        'description': 'Unique identifier for customers',
        'type': 'qualitative',
        'subtype': 'identifier',
    },
    {
        'variable': 'Vehicle Type',
        'description': 'Type of vehicle (Go Mini, Go Sedan, Auto, eBike/Bike, UberXL, Premier Sedan',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
    {
        'variable': 'Pickup Location',
        'description': 'Starting location of the ride',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
     {
        'variable': 'Drop Location',
        'description': 'Destination location of the ride',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
     {
        'variable': 'Avg VTAT',
        'description': 'Average time for driver to reach pickup location (in minutes)',
        'type': 'quantitative',
        'subtype': 'continuous',
    },
    {
        'variable': 'Avg CTAT',
        'description': 'Average trip duration from pickup to destination (in minutes)',
        'type': 'quantitative',
        'subtype': 'continuous',
    },
     {
        'variable': 'Cancelled Rides by Customer',
        'description': 'Customer-initiated cancellation flag',
        'type': 'quantitative',
        'subtype': 'discrete',
    },
    {
        'variable': 'Reason for cancelling by Customer',
        'description': 'Reason for customer cancellation',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
     {
        'variable': 'Cancelled Rides by Driver',
        'description': 'Driver-initiated cancellation flag',
        'type': 'quantitative',
        'subtype': 'discrete',
    },
    {
        'variable': 'Driver Cancellation Reason',
        'description': 'Reason for driver cancellation',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
    {
        'variable': 'Incomplete Rides',
        'description': 'Incomplete ride flag',
        'type': 'quantitative',
        'subtype': 'discrete',
    },
    {
        'variable': 'Incomplete Rides Reason',
        'description': 'Reason for incomplete rides',
        'type': 'qualitative',
        'subtype': 'nominal',
    },
    {
        'variable': 'Booking Value',
        'description': 'Total fare amount for the ride',
        'type': 'quantitative',
        'subtype': 'continua',
    },
    {
        'variable': 'Ride Distance',
        'description': 'Distance covered during the ride (in km)',
        'type': 'quantitative',
        'subtype': 'continua',
    },
    {
        'variable': 'Driver Ratings',
        'description': 'Rating given to driver (1-5 scale)',
        'type': 'quantitative',
        'subtype': 'discrete',
    },
     {
        'variable': 'Customer Rating',
        'description': 'Rating given by customer (1-5 scale)',
        'type': 'quantitative',
        'subtype': 'discrete',
    },
     {
        'variable': 'Payment Method',
        'description': 'Method used for payment (UPI, Cash, Credit Card, Uber Wallet, Debit Card)',
        'type': 'qualitative',
        'subtype': 'nominal',
    }
])
    

In [10]:
data_dictionary

,variable,description,type,subtype
0,Date,Date of the booking,temporal,date
1,Time,Time of the booking,temporal,time
2,Booking ID,Unique identifier for each ride booking,qualitative,identifier
3,Booking Status,"Status of booking (Completed, Cancelled by Cus...",qualitative,nominal
4,Customer ID,Unique identifier for customers,qualitative,identifier
5,Vehicle Type,"Type of vehicle (Go Mini, Go Sedan, Auto, eBik...",qualitative,nominal
6,Pickup Location,Starting location of the ride,qualitative,nominal
7,Drop Location,Destination location of the ride,qualitative,nominal
8,Avg VTAT,Average time for driver to reach pickup locati...,quantitative,continuous
9,Avg CTAT,Average trip duration from pickup to destinati...,quantitative,continuous


In [11]:
# salvar o dicionario
data_dictionary.to_csv('../data/processed/data_dictionary.csv', index=False)

## 6. Limpeza e tratamento dos dados 

In [12]:
# verificar dados duplicados
df.duplicated().sum()

np.int64(0)

In [13]:
# verificar dados faltantes
df.isna().sum()

Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10500
Avg CTAT                              48000
Cancelled Rides by Customer          139500
Reason for cancelling by Customer    139500
Cancelled Rides by Driver            123000
Driver Cancellation Reason           123000
Incomplete Rides                     141000
Incomplete Rides Reason              141000
Booking Value                         48000
Ride Distance                         48000
Driver Ratings                        57000
Customer Rating                       57000
Payment Method                        48000
dtype: int64

In [14]:
# verificar os dados faltantes da coluna Avg VTAT 
df['Avg VTAT'].isnull().sum()

np.int64(10500)

In [15]:
df['Avg VTAT'].describe()

count    139500.000000
mean          8.456352
std           3.773564
min           2.000000
25%           5.300000
50%           8.300000
75%          11.300000
max          20.000000
Name: Avg VTAT, dtype: float64

In [16]:
df['Avg VTAT'].fillna(df['Avg VTAT'].mean(), inplace=True)

In [17]:
# verificar os dados faltantes da coluna Avg VTAT 
df['Avg CTAT'].isnull().sum()

np.int64(48000)

In [18]:
df['Avg CTAT'].describe()

count    102000.000000
mean         29.149636
std           8.902577
min          10.000000
25%          21.600000
50%          28.800000
75%          36.800000
max          45.000000
Name: Avg CTAT, dtype: float64

In [19]:
df['Avg CTAT'].fillna(df['Avg CTAT'].mean(), inplace=True)

In [20]:
# verificar os dados faltantes da coluna Cancelled Rides by Customer
df['Cancelled Rides by Customer'].isna().sum()

np.int64(139500)

In [21]:
df['Cancelled Rides by Customer'].describe()

count    10500.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: Cancelled Rides by Customer, dtype: float64

In [22]:
df['Cancelled Rides by Customer'].unique()

array([nan,  1.])

In [23]:
# preencher os valores ausentes com 0(zero)
df['Cancelled Rides by Customer'].fillna(0, inplace=True)

In [24]:
df['Cancelled Rides by Customer'].unique()

array([0., 1.])

In [25]:
# verificar os dados faltantes da coluna Reason for cancelling by Customer
df['Reason for cancelling by Customer'].isnull().sum()

np.int64(139500)

In [26]:
# os motivos do cancelamentos por parte do cliente
df['Reason for cancelling by Customer'].unique()

array([nan, 'Driver is not moving towards pickup location',
       'Driver asked to cancel', 'AC is not working', 'Change of plans',
       'Wrong Address'], dtype=object)

In [27]:
df['Reason for cancelling by Customer'].value_counts(dropna=False)

Reason for cancelling by Customer
NaN                                             139500
Wrong Address                                     2362
Change of plans                                   2353
Driver is not moving towards pickup location      2335
Driver asked to cancel                            2295
AC is not working                                 1155
Name: count, dtype: int64

In [28]:
# preencher com No Reason(sem motivo) os nan
df['Reason for cancelling by Customer'] = df['Reason for cancelling by Customer'].fillna('No Reason')

In [29]:
df['Reason for cancelling by Customer'].value_counts(dropna=False)

Reason for cancelling by Customer
No Reason                                       139500
Wrong Address                                     2362
Change of plans                                   2353
Driver is not moving towards pickup location      2335
Driver asked to cancel                            2295
AC is not working                                 1155
Name: count, dtype: int64

In [30]:
## verificar os dados faltantes da coluna Cancelled Rides by Driver
df['Cancelled Rides by Driver'].isnull().sum()

np.int64(123000)

In [31]:
df['Cancelled Rides by Driver'].unique()

array([nan,  1.])

In [32]:
# preencher os valores ausentes com 0(zero)
df['Cancelled Rides by Driver'].fillna(0, inplace=True)

In [33]:
df['Cancelled Rides by Driver'].unique()

array([0., 1.])

In [34]:
## verificar os dados faltantes da coluna Driver Cancellation Reason
df['Driver Cancellation Reason'].isnull().sum()

np.int64(123000)

In [35]:
df['Driver Cancellation Reason'].value_counts(dropna=False)

Driver Cancellation Reason
NaN                                    123000
Customer related issue                   6837
The customer was coughing/sick           6751
Personal & Car related issues            6726
More than permitted people in there      6686
Name: count, dtype: int64

In [36]:
# preencher com No Reason(sem motivo) os nan
df['Driver Cancellation Reason'] = df['Driver Cancellation Reason'].fillna('No Reason')

In [37]:
df['Driver Cancellation Reason'].value_counts(dropna=False)

Driver Cancellation Reason
No Reason                              123000
Customer related issue                   6837
The customer was coughing/sick           6751
Personal & Car related issues            6726
More than permitted people in there      6686
Name: count, dtype: int64

In [38]:
# verificar os dados faltantes da coluna Incomplete Rides
df['Incomplete Rides'].isnull().sum()

np.int64(141000)

In [39]:
df['Incomplete Rides'].value_counts(dropna=False)

Incomplete Rides
NaN    141000
1.0      9000
Name: count, dtype: int64

In [40]:
df['Incomplete Rides'].fillna(0, inplace=True)

In [41]:
df['Incomplete Rides'].value_counts(dropna=False)

Incomplete Rides
0.0    141000
1.0      9000
Name: count, dtype: int64

In [42]:
# verificar os dados faltantes da coluna Incomplete Rides Reason
df['Incomplete Rides Reason'].isnull().sum()

np.int64(141000)

In [43]:
df['Incomplete Rides Reason'].value_counts(dropna=False)

Incomplete Rides Reason
NaN                  141000
Customer Demand        3040
Vehicle Breakdown      3012
Other Issue            2948
Name: count, dtype: int64

In [44]:
# preencher com No Reason(sem motivo) os nan
df['Incomplete Rides Reason'] = df['Incomplete Rides Reason'].fillna('No Reason')

In [45]:
## verificar os dados faltantes da coluna Booking Value
df['Booking Value'].isnull().sum()

np.int64(48000)

In [46]:
df['Booking Value'].describe()

count    102000.000000
mean        508.295912
std         395.805774
min          50.000000
25%         234.000000
50%         414.000000
75%         689.000000
max        4277.000000
Name: Booking Value, dtype: float64

In [47]:
# verificar se a corrida foi cancelada podemos verificar pela variavel Booking Status
df['Booking Status'].value_counts()

Booking Status
Completed                93000
Cancelled by Driver      27000
No Driver Found          10500
Cancelled by Customer    10500
Incomplete                9000
Name: count, dtype: int64

In [48]:
## Verificar os valores faltantes por status da corrida
df.groupby('Booking Status')['Booking Value'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    10500
Cancelled by Driver      27000
Completed                    0
Incomplete                   0
No Driver Found          10500
Name: Booking Value, dtype: int64

**Valores ausentes na variável Booking Value estão associados a corridas não concluídas (canceladas ou sem motorista). Como essas corridas não geram receita, os valores ausentes foram substituídos por 0 para representar ausência de faturamento.**

In [49]:
# preencher com 0(zero) os valores ausentes
df['Booking Value'] = df['Booking Value'].fillna(0)

In [50]:
df['Booking Value'].describe()

count    150000.000000
mean        345.641220
std         403.423487
min           0.000000
25%           0.000000
50%         244.000000
75%         521.000000
max        4277.000000
Name: Booking Value, dtype: float64

In [51]:
# verificar os valores ausentes da coluna Ride Distance
df['Ride Distance'].isnull().sum()

np.int64(48000)

In [52]:
df['Ride Distance'].describe()

count    102000.000000
mean         24.637012
std          14.002138
min           1.000000
25%          12.460000
50%          23.720000
75%          36.820000
max          50.000000
Name: Ride Distance, dtype: float64

In [53]:
## Verificar os valores faltantes por status da corrida
df.groupby('Booking Status')['Ride Distance'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    10500
Cancelled by Driver      27000
Completed                    0
Incomplete                   0
No Driver Found          10500
Name: Ride Distance, dtype: int64

In [54]:
# preencher com 0(zero) os valores ausentes pelo mesmo motivo anterior se a corrida foi cancelada não tem deslocamento
df['Ride Distance'] = df['Ride Distance'].fillna(0)

In [55]:
## verificar os dados faltantes da coluna Driver Ratings
df['Driver Ratings'].isnull().sum()

np.int64(57000)

In [56]:
df['Driver Ratings'].value_counts(dropna=False)

Driver Ratings
NaN    57000
4.3    14081
4.2    13841
4.6     9368
4.4     7018
4.1     6966
4.9     4705
4.7     4678
4.5     4634
3.9     3915
3.8     3848
3.7     3790
5.0     2365
4.8     2328
3.6     2026
4.0     1995
3.2     1538
3.4     1491
3.3     1461
3.1     1459
3.5      748
3.0      745
Name: count, dtype: int64

In [57]:
df['Driver Ratings'].describe()

count    93000.000000
mean         4.230992
std          0.436871
min          3.000000
25%          4.100000
50%          4.300000
75%          4.600000
max          5.000000
Name: Driver Ratings, dtype: float64

In [58]:
# verificar o status da corrida se foi cancelada não possui avaliação
df.groupby('Booking Status')['Driver Ratings'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    10500
Cancelled by Driver      27000
Completed                    0
Incomplete                9000
No Driver Found          10500
Name: Driver Ratings, dtype: int64

In [59]:
## preencher com a mediana neste caso
median_rating = df['Driver Ratings'].median()

df['Driver Ratings'] = df['Driver Ratings'].fillna(median_rating)

In [60]:
df.groupby('Booking Status')['Driver Ratings'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    0
Cancelled by Driver      0
Completed                0
Incomplete               0
No Driver Found          0
Name: Driver Ratings, dtype: int64

In [61]:
# verificar os valores ausentes da coluna Customer Rating
df['Customer Rating'].isnull().sum()

np.int64(57000)

In [62]:
# verificar o status da corrida se foi cancelada não possui avaliação
df.groupby('Booking Status')['Customer Rating'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    10500
Cancelled by Driver      27000
Completed                    0
Incomplete                9000
No Driver Found          10500
Name: Customer Rating, dtype: int64

In [63]:
df['Customer Rating'].describe()

count    93000.000000
mean         4.404584
std          0.437819
min          3.000000
25%          4.200000
50%          4.500000
75%          4.800000
max          5.000000
Name: Customer Rating, dtype: float64

In [64]:
# usar o mesmo criterio do anterior
## preencher com a mediana neste caso
median_rating = df['Customer Rating'].median()

df['Customer Rating'] = df['Customer Rating'].fillna(median_rating)

In [65]:
df.groupby('Booking Status')['Customer Rating'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    0
Cancelled by Driver      0
Completed                0
Incomplete               0
No Driver Found          0
Name: Customer Rating, dtype: int64

In [66]:
## Payment Method 
df['Payment Method'].isnull().sum()

np.int64(48000)

In [67]:
df.groupby('Booking Status')['Payment Method'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    10500
Cancelled by Driver      27000
Completed                    0
Incomplete                   0
No Driver Found          10500
Name: Payment Method, dtype: int64

In [68]:
# preencher com 0(zero) os valores ausentes
df['Payment Method'] = df['Payment Method'].fillna(0)

In [69]:
df.groupby('Booking Status')['Payment Method'].apply(lambda x: x.isna().sum())

Booking Status
Cancelled by Customer    0
Cancelled by Driver      0
Completed                0
Incomplete               0
No Driver Found          0
Name: Payment Method, dtype: int64

### 6.1. Regras utilizadas para imputação de valores ausentes

Durante o processo de limpeza dos dados foram aplicadas diferentes estratégias de imputação, considerando o tipo da variável e o significado dos valores ausentes no contexto do dataset.

**1. Variáveis quantitativas contínuas**

**Variáveis:** `Avg VTAT`, `Avg CTAT`

**Estratégia:** preenchimento com média da variável.

**Justificativa:**

Essas variáveis representam tempos médios associados à corrida. A utilização da média permite preencher os valores ausentes preservando a distribuição geral dos dados e evitando distorções significativas nas análises posteriores.

---

**2. Variáveis quantitativas discretas**

**Variáveis:** `Cancelled Rides by Customer`, `Cancelled Rides by Driver`, `Incomplete Rides`

**Estratégia:** preenchimento com 0 (zero).

**Justificativa:**
Valores ausentes nessas variáveis indicam que o evento não ocorreu. Assim, o valor 0 representa corretamente a ausência de cancelamento ou interrupção da corrida.

---

**3. Variáveis qualitativas nominais**

**Variáveis:** `Reason for cancelling by Customer`, `Driver Cancellation Reason`,`Incomplete Rides Reason`

**Estratégia:** preenchimento com a categoria "No Reason".

**Justificativa:**
Valores ausentes nessas variáveis indicam ausência de motivo registrado. A criação da categoria "No Reason" permite preservar a integridade da variável categórica e facilita futuras análises exploratórias.

--- 
**4. Variáveis relacionadas ao valor da corrida**

**Variáveis:** `Booking Value`, `Ride Distance`

**Estratégia:** preenchimento com 0 (zero).

**Justificativa:**
Os valores ausentes nessas variáveis estão associados a corridas que não foram concluídas, como casos de cancelamento ou ausência de motorista. Como nessas situações não há faturamento nem deslocamento, os valores foram substituídos por 0, representando corretamente a ausência de receita e distância percorrida.

---

**5. Variáveis de avaliação**

**Variáveis:** `Driver Ratings`, `Customer Rating`

**Estratégia:** preenchimento com mediana da variável.

**Justificativa:**
Essas variáveis representam avaliações em escala numérica. A utilização da mediana reduz o impacto de possíveis assimetrias ou valores extremos, preservando melhor a distribuição dos dados.

---


### Tabela resumo das estratégias de imputação dos dados ausentes

In [72]:
handling_missing_values = pd.DataFrame([
    {
        'Variável': 'Avg VTAT',
        'Tipo': 'Quantitativa contínua',
        'Estratégia': 'Média',
        'Justificativa': 'Representa tempo médio até o motorista chegar. A média preserva a distribuição da variável.',
    },
    {
        'Variável': 'Avg CTAT',
        'Tipo': 'Quantitativa contínua',
        'Estratégia': 'Média',
        'Justificativa': 'Representa duração média da corrida. A média mantém a consistência dadistribuição.',
    },
    {
        'Variável': 'Cancelled Rides by Customer',
        'Tipo': 'Quantitativa discreta',
        'Estratégia': '0(zero)',
        'Justificativa': 'Valor ausente indica que não houve cancelamento pelo cliente.',
    },
     {
        'Variável': 'Cancelled Rides by Driver',
        'Tipo': 'Quantitativa discreta',
        'Estratégia': '0(zero)',
        'Justificativa': 'Valor ausente indica ausência de cancelamento pelo motorista.',
    },
    {
        'Variável': 'Incomplete Rides',
        'Tipo': 'Quantitativa discreta',
        'Estratégia': '0(zero)',
        'Justificativa': 'Valor ausente indica que a corrida não foi interrompida.',
    },
    {
        'Variável': 'Reason for cancelling by Customer',
        'Tipo': 'Quantitativa nominal',
        'Estratégia': '"No Reason"',
        'Justificativa': 'Indica ausência de motivo registrado para cancelamento.',
    },
    {
        'Variável': 'Driver Cancellation Reason',
        'Tipo': 'Quantitativa nominal',
        'Estratégia': '"No Reason"',
        'Justificativa': 'Mantém a integridade da variável categórica.',
    },
     {
        'Variável': 'Incomplete Rides Reason',
        'Tipo': 'Quantitativa nominal',
        'Estratégia': '"No Reason"',
        'Justificativa': 'Indica ausência de motivo para interrupção da corrida.',
    },
     {
        'Variável': 'Booking Value',
        'Tipo': 'Quantitativa contínua',
        'Estratégia': '0(zero)',
        'Justificativa': 'Corridas canceladas ou não realizadas não geram receita.',
    },
     {
        'Variável': 'Ride Distance',
        'Tipo': 'Quantitativa contínua',
        'Estratégia': '0(zero)',
        'Justificativa': 'Corridas não realizadas não possuem distância percorrida.',
    },
     {
        'Variável': 'Driver Ratings',
        'Tipo': 'Quantitativa contínua',
        'Estratégia': 'Mediana',
        'Justificativa': 'Reduz impacto de possíveis assimetrias na distribuição.',
    },
    {
        'Variável': 'Customer Rating',
        'Tipo': 'Quantitativa contínua',
        'Estratégia': 'Mediana',
        'Justificativa': 'Mantém estabilidade da distribuição das avaliações.',
    }
    ])

In [73]:
handling_missing_values

,Variável,Tipo,Estratégia,Justificativa
0,Avg VTAT,Quantitativa contínua,Média,Representa tempo médio até o motorista chegar....
1,Avg CTAT,Quantitativa contínua,Média,Representa duração média da corrida. A média m...
2,Cancelled Rides by Customer,Quantitativa discreta,0(zero),Valor ausente indica que não houve cancelament...
3,Cancelled Rides by Driver,Quantitativa discreta,0(zero),Valor ausente indica ausência de cancelamento ...
4,Incomplete Rides,Quantitativa discreta,0(zero),Valor ausente indica que a corrida não foi int...
5,Reason for cancelling by Customer,Quantitativa nominal,"""No Reason""",Indica ausência de motivo registrado para canc...
6,Driver Cancellation Reason,Quantitativa nominal,"""No Reason""",Mantém a integridade da variável categórica.
7,Incomplete Rides Reason,Quantitativa nominal,"""No Reason""",Indica ausência de motivo para interrupção da ...
8,Booking Value,Quantitativa contínua,0(zero),Corridas canceladas ou não realizadas não gera...
9,Ride Distance,Quantitativa contínua,0(zero),Corridas não realizadas não possuem distância ...


In [74]:
# salvar o dicionario
data_dictionary.to_csv('../data/processed/handling_missing_values.csv', index=False)

In [70]:

# verificar dados faltantes
df.isna().sum()

Date                                 0
Time                                 0
Booking ID                           0
Booking Status                       0
Customer ID                          0
Vehicle Type                         0
Pickup Location                      0
Drop Location                        0
Avg VTAT                             0
Avg CTAT                             0
Cancelled Rides by Customer          0
Reason for cancelling by Customer    0
Cancelled Rides by Driver            0
Driver Cancellation Reason           0
Incomplete Rides                     0
Incomplete Rides Reason              0
Booking Value                        0
Ride Distance                        0
Driver Ratings                       0
Customer Rating                      0
Payment Method                       0
dtype: int64

## 7. Engenharia de Variáveis (Feature Engineering)

### Insight
- muitas corridas cancelada pelo Motorista
- muitas corridas cancelada pelo clientes
- muitas cooridas com status de Nenhum motorista encontrado 
Isso pode indica:
- falta de motoristas em certas regiões
- corridas curtas que motoristas não querem aceitar
- problemas de preço
- cliente cancela por engano


### Algumas perguntas que pode ser respondidas com este dataset
**Perguntas como:**
- Qual a taxa de cancelamento?
- Onde faltam motoristas?
- Quais veículos geram mais receitas?
- Quais regiões têm mais corridas?
- Quais os motivos dos cancelamento tanto dos motoristas quanto dos clientes?
- Quais os horários de maior demanda?
- Quais dias da semana possui possui maior demanda?
- Corridas curtas são mais frequentes?
- 

